# Part 1: Build a deep research agent

In this part, you will implement a deep research agent in LangGraph and analyze the requests it sends to a language model. The agent divides a question into subtopics, researches them in parallel, and writes a report with citations to source passages.

In Assignment 3, each simulated request represented a single LLM call. In an agentic workload, later LLM calls may wait for earlier calls or tools to finish, and related calls often share long prompt prefixes. You will use the recorded traces to study these execution dependencies, prefix reuse, and the effect of concurrency. Parts 2 and 3 examine how to schedule and cache this workload.

Complete the following:

- `[Part 1] TODO` sections in `research_app/concurrency.py`: Implement `acquire`, `release`, and `set_limit` to control how many LLM calls can run at once across all reports.
- `[Part 1] TODO` sections in `research_app/agent.py`: Implement the graph nodes, routing functions, and `build_graph` so the agent can plan its research, gather evidence, write a report, and check its citations.
- `shared_prefix_share` in section 5b of this notebook: Calculate the fraction of prompt tokens in prefix blocks that appeared in earlier calls.

Run the notebook cells to check your implementation and produce plots for your report. See the [Part 1 handout](../docs/part1_handout.md) for the full specification.

| Section | Task | Estimated time |
|---|---|---|
| 1. Corpus and tools | Inspect the corpus, search results, and questions | 15 min |
| 2. Concurrency cap | Implement `AdjustableCap` | 30 min |
| 3. Research agent | Implement the graph | 2–3 h |
| 4. Run the agent | Run 16 reports at C = 2 and C = 8 | 15 min |
| 5. Workload analysis | Analyze dependencies, prefix sharing, and concurrency | 1 h |

**Estimated time:** 4–5 hours.

**Grading (15 points):** implementation correctness (9 points); workload analysis (6 points).

## Before you begin

Complete the [environment setup](../README.md#getting-started) before running this notebook. Select **CS 378 Assignment 4** as the notebook kernel so its cells use the `.venv` environment where you installed the assignment dependencies. (When you open a new terminal, run `source .venv/bin/activate` from the repository root; activating `.venv` in a terminal does not change the kernel of an already-open notebook.)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())
plt.rcParams["figure.dpi"] = 100

BACKEND = "bedrock" if os.environ.get("AGENTKV_BEDROCK_MODEL") else "fake"
print(f"Working in {Path.cwd()}")
hint = "" if BACKEND == "bedrock" else "  (set AGENTKV_BEDROCK_MODEL to use Bedrock)"
print(f"LLM backend: {BACKEND}{hint}")

## 1. Corpus and tools

Before implementing the agent, you will explore the offline corpus of 48 Wikipedia articles on six systems topics (each saved at two revisions, `v1` and `v2`). Rather than querying the live web, we supply the following two tools over this corpus:

- `search(query, k=5)` ranks articles by BM25 relevance and returns up to `k` matching `SearchHit` results, each containing the `doc_id`, title, relevance score, snapshot hash, and a short snippet.
- `fetch(doc_id)` retrieves the full article text and revision metadata (`Doc`) for a given `doc_id`.

When called through the agent's tool wrapper, both tools also simulate reproducible delays and occasional transient failures using a seeded latency model so you can examine how tool calls affect the agent's progress.

Use this local corpus while implementing the graph and running the controlled experiments in sections 4 and 5. After completing the required work, you can also connect your agent to live web search in section 6.

In [ ]:
from research_app.corpus import Corpus
from research_app.questions import load_questions

corpus = Corpus("v2")
print({topic: len(ids) for topic, ids in corpus.topics().items()})

for hit in corpus.search("how does raft elect a leader", k=3):
    print(f"{hit.doc_id:40} score {hit.score:.2f}  {hit.title}")

In [ ]:
doc = corpus.fetch(corpus.search("how does raft elect a leader", k=1)[0].doc_id)
print(doc.title, f"({len(doc.text)} characters, snapshot sha256 {doc.sha256[:12]})")
print(doc.text[:600], "...")

The next cell lists the first five of the 16 public research questions. Try passing a few queries to `corpus.search()` and inspect the results; if a query does not retrieve the article you expected, try rephrasing it to see how BM25 ranking responds.

In [ ]:
questions = load_questions(public=True)
for q in questions[:5]:
    print(f"{q['id']:18} {q['question']}")
print(f"... {len(questions)} questions")

## 2. Concurrency cap

Across all active reports, the application enforces a shared limit **C** on how many LLM calls can run at once (in Parts 2 and 3, serving policies will control the corresponding limit via `target_concurrency`).

Complete the following methods on `AdjustableCap` in `research_app/concurrency.py`:

- `acquire()`: Wait until `in_flight < limit`, reserve a slot, and update the counters for current in-flight calls, total grants, and peak concurrency.
- `release()`: Return a slot and wake waiting calls so they can re-check whether a slot is available.
- `set_limit(n)`: Update the limit and wake waiting calls. If the limit is lowered below `in_flight`, let running calls finish without interruption while keeping new calls waiting until `in_flight < limit`.

The supplied `slot()` context manager calls `acquire()` and `release()` around an entire LLM operation (including retries, so a retry keeps its existing slot). Use an `asyncio.Condition` to coordinate waiting coroutines, then run the unit test cell below.

In [ ]:
check("tests/test_part1_concurrency.py")

The next cell tests how your limiter responds when the cap changes mid-run. Six calls start with an initial limit of 3, and while the first three are running, the limit drops to 1:

- **Upper plot (per-call timeline):** Yellow bars show time waiting for a slot; blue bars show time running.
- **Lower plot (concurrency over time):** Shows the number of running calls alongside the active limit (dashed line).

After the limit drops to 1, the three running calls should finish normally (so `in_flight` temporarily exceeds the new limit without counting as an error), and the remaining three calls should then run one at a time.

In [ ]:
from research_app.concurrency import AdjustableCap
from research_app.visualize import run_cap_demo, plot_cap_demo

cap = AdjustableCap(3)
demo = await run_cap_demo(cap)
plot_cap_demo(demo)
plt.show()
print(f"Peak running calls: {demo.peak} (expected 3)")
print(f"Calls incorrectly started at or above the limit: {demo.over_limit} (expected 0)")

## 3. Research agent

Next, implement the research agent in `research_app/agent.py`. The workflow below shows how the agent plans subtopics, researches them in parallel, checks subtopic coverage (running at most one follow-up branch if needed), synthesizes a cited report, and validates citations (with at most one repair call):

![Research graph: plan, parallel research branches, coverage check, optional follow-up, synthesize, validate, and optional repair.](../docs/figures/research_workflow.svg)

See the [Part 1 handout](../docs/part1_handout.md) for a quick overview of the LangGraph primitives (`StateGraph`, edges, conditional edges, `Send` fan-out, and join) and the required `op_id` and `order` conventions. Keep three rules in mind as you implement the nodes:

- **Route all calls through `ctx.client`:** The tracing client records the trace and enforces per-report budgets (at most 8 LLM calls, 5 searches, 15 fetches, and 3 attempts per operation).
- **Record exact `deps` and use `research_app/prompts.py`:** Each operation's `deps` list defines the edges of the exported DAG (an `extract` call depends on all fetches in its branch, including failed ones, though only fetched documents enter its prompt). The prompt builders place shared instructions and documents first so calls share prefixes.
- **Handle permanent tool failures gracefully:** A call that fails permanently returns `None`; return a typed `BranchResult` rather than raising an exception so the rest of the report can proceed.

Start by implementing `build_graph`, then fill in the nodes and routing functions. The cell below runs a single report through `ResearchAgent.research(topic)` and stops with `NotImplementedError` at the first unfinished node.

In [ ]:
from IPython.display import Markdown, display
from research_app import ResearchAgent
from research_app.llm import make_llm
from research_app.search import LocalSearch

agent = ResearchAgent(llm=make_llm(BACKEND), search=LocalSearch(), cap=2)
result = await agent.research(questions[0]["question"])
print(f"Citation checks passed: {result.valid}; LLM calls: {result.recording.counts()['llm_calls']}")
print("Errors:", result.recording.errors)

In [ ]:
display(Markdown(result.markdown()))

In [ ]:
check("tests/test_part1_agent.py", "tests/test_part1_citations.py", "tests/test_part1_llm.py",
      "tests/test_part1_trace_export.py")

## 4. Run the agent at C = 2 and C = 8

Now run all 16 public questions at two concurrency caps—`C = 2` (`runs/c2/`) and `C = 8` (`runs/c8/`)—launching one new report every 2 seconds.

These runs use the Part 1 backend selected in the setup cell (`bedrock` if `AGENTKV_BEDROCK_MODEL` is set, otherwise `fake`). With `FakeLLM`, the `C = 2` run takes ~90 seconds and `C = 8` takes under a minute; Bedrock runs take a few minutes. Note which backend you used in your report. (Unlike Parts 2 and 3, which replay fixed traces in the Vidur GPU simulator, Part 1 runs your agent directly.)

Each run writes four files to its output directory:

- `trace.jsonl`: Every LLM and tool event with timestamps and prompt `prefix_ids`.
- `reports.json`: The generated reports and citations.
- `summary.json`: Aggregate latency, concurrency, and citation-validation stats.
- `replay.json`: The run exported in the DAG trace format used by Parts 2 and 3.

Verify that both runs report `16/16 valid` in `summary.json` before moving on.

In [ ]:
for C in (2, 8):
    !{sys.executable} -m research_app.run --backend {BACKEND} --cap {C} --replay --out runs/c{C}

## 5. Workload analysis

Using `runs/c2/` and `runs/c8/`, you will now analyze the agent's execution DAG, shared prompt prefixes, and concurrency behavior, then answer Q1–Q3 below.

### 5a. Dependencies and critical path

The plot below shows the recorded DAG for one report (`R00`). Each box displays an operation and its measured service time (including any retry backoff, but excluding time spent waiting for an LLM concurrency slot). Red arrows highlight the **critical path**—the dependency chain with the longest total service time. Compare this DAG with the timeline in section 5c to see how slot queueing affects total report completion time.

In [ ]:
from research_app.visualize import plot_research_dag

# R00 is the first report; choose another report id to inspect a different run.
fig, critical_ops = plot_research_dag("runs/c8/trace.jsonl", report_id="R00")
plt.show()
print("Longest service-time path:")
print(" → ".join(critical_ops))

### 5b. Shared prompt prefixes

Each completed LLM call in `trace.jsonl` records `prefix_ids` for its full 256-token prompt blocks (each ID hashes the prefix up to that block, so two calls share a block ID only if their prompts are identical through that block). Partial trailing blocks (< 256 tokens) have no ID and are never counted as shared, though they are included in `prompt_tokens`.

Count a full 256-token block as **shared** if its ID appeared in any earlier LLM call (processing calls in start order across all stages). This measures potential prefix reuse; whether a block is still in memory when the later call runs depends on cache capacity and eviction (studied in Parts 2 and 3).

Implement `shared_prefix_share` below, then run the cell to compute the overall and per-stage shared-prefix fractions at `C = 2` and `C = 8`.

In [ ]:
def llm_calls(trace_path):
    """Return completed LLM call events in start order, including each call's stage."""
    events = [json.loads(line) for line in Path(trace_path).read_text().splitlines()]
    starts = {e["op_id"]: e for e in events if e["event"] == "op_start"}
    calls = [{**e, "stage": starts[e["op_id"]]["stage"]}
             for e in events if e["event"] == "op_end" and "prefix_ids" in e]
    return sorted(calls, key=lambda e: starts[e["op_id"]]["t"])


def shared_prefix_share(calls, block_size=256):
    """Calculate the fraction of prompt tokens in previously seen prefix blocks."""
    # [Part 1] TODO: Measure how much prompt content appeared in earlier calls.
    # calls is already in start order. A full prefix block counts as shared
    # only if its id appeared in a previous call; add this call's ids to your
    # history after counting its shared blocks. Each shared block contributes
    # block_size tokens. Divide the total shared tokens by the sum of
    # prompt_tokens, which includes partial final blocks. Return 0.0 when
    # there are no prompt tokens. Keep one history across all stages.
    # Use: calls, call["prefix_ids"], call["prompt_tokens"], block_size.
    raise NotImplementedError("Part 1: fill shared_prefix_share")
    # END OF YOUR CODE


from research_app.visualize import plot_prefix_reuse

for C in (2, 8):
    calls = llm_calls(f"runs/c{C}/trace.jsonl")
    print(f"C = {C}: {len(calls)} LLM calls; shared prefix share {shared_prefix_share(calls):.1%}")
    plot_prefix_reuse(calls, shared_prefix_share)
    plt.show()

### 5c. Effect of the concurrency cap

The cells below compare `C = 2` and `C = 8` using summary bar charts (mean report latency, total makespan, and mean slot-wait time per LLM call) and per-report timelines:

- **Yellow:** LLM call waiting for a concurrency slot.
- **Blue:** LLM call holding a slot (running or in retry backoff).
- **Gray:** Tool execution (`search` or `fetch`).
- **Black tick (`|`):** Report arrival time.

Note that Part 1 runs do not model GPU KV-cache capacity or cache eviction, so raising `C` from 2 to 8 reduces slot queueing without incurring memory-pressure preemptions. In Parts 2 and 3, you will see what happens when concurrent calls compete for a fixed KV cache.

In [ ]:
from research_app.visualize import plot_run_comparison, plot_run_timelines

run_dirs = ["runs/c2", "runs/c8"]
fig, measurements = plot_run_comparison(run_dirs)
plt.show()
plot_run_timelines(run_dirs)
plt.show()

for row in measurements:
    print(f"C = {row['cap']}: {row['valid']}/{row['reports']} valid reports; "
          f"peak {row['llm_peak']} concurrent LLM calls; "
          f"mean wait for a slot {row['mean_slot_wait']:.2f} s per LLM call")

## Questions

Answer Q1–Q3 below, keeping each question above its answer. Include your answers, plots, and completed table in `assignment4_report.pdf`, in the same order, and include both `runs/c2/` and `runs/c8/` in your Canvas archive (see [Submission](../README.md#submitting)).

**Q1.** Which operations are on the critical path of the report you examined? Include its dependency graph and explain which operations determine when it finishes. How much time does the report spend waiting for an LLM slot at C = 2?

**Your answer:**

**Q2.** What fraction of prompt tokens belong to prefix blocks that appeared in earlier calls? Use `prefix_ids` and `prompt_tokens`; only full 256-token blocks have prefix ids, so the final partial block of a prompt does not count as shared. Report the fraction by stage and explain what each stage reuses. Which prefixes would be useful to retain between calls, and for how long? Use measurements from both runs to support your answer.

**Your answer:**

**Q3.** How does increasing the concurrency cap from C = 2 to C = 8 affect mean report latency, makespan, waiting time, and peak concurrent calls? Include the timelines and a table comparing the two runs. Estimate the GPU memory needed at each cap, accounting for shared prefixes. For this estimate, assume Llama-3-8B in bf16 on one 80 GB A100, with approximately 16 GB of weights and 128 KiB of KV cache per token (32 layers × 8 KV heads × 128 dimensions × 2 for K and V × 2 bytes). Explain the tradeoff between memory use and completion time.

**Your answer:**

## 6. [Optional] Research a topic on the web

Once you have finished the required implementation and analysis above, you can use your `ResearchAgent` to investigate any topic on the live web. This extension is optional and does not affect your grade.

Keep the same `ResearchAgent` with `BedrockLLM()` and replace `LocalSearch()` with one of the web-search adapters supplied in `research_app/search.py` (`"exa"`, `"tavily"`, `"perplexity"`, or `"openai"`). See the [Artificial Analysis Search API leaderboard](https://artificialanalysis.ai/agents/search-api) for a comparison of search providers, and the [Part 1 handout](../docs/part1_handout.md#optional-research-a-topic-on-the-web) for details on how each adapter retrieves and archives source text:

| Provider | Adapter and environment variable | Free access (checked October 30, 2026) |
|---|---|---|
| [Exa](https://exa.ai/pricing) | `ExaSearch`, `EXA_API_KEY` | $10 signup bonus plus $10/month; no card required. |
| [Tavily](https://www.tavily.com/pricing) | `TavilySearch`, `TAVILY_API_KEY` | 1,000 credits/month (or 4,000/month via [student offer](https://help.tavily.com/articles/6606514713-student-account)); no card required. |
| [Perplexity Search](https://docs.perplexity.ai/docs/getting-started/pricing) | `PerplexitySearch`, `PERPLEXITY_API_KEY` | Pay as you go. |
| [OpenAI web search](https://developers.openai.com/api/docs/guides/tools-web-search) | `OpenAISearch`, `OPENAI_API_KEY`, `AGENTKV_SEARCH_MODEL` | Pay as you go (also set `AGENTKV_SEARCH_MODEL`). |

Configure Bedrock ([Model access](../README.md#model-access)) and your search provider's API key in the environment before launching Jupyter, then set `RUN_WEB_RESEARCH = True` below.

In [ ]:
from datetime import datetime
from research_app import ResearchAgent
from research_app.llm import BedrockLLM
from research_app.search import make_search

RUN_WEB_RESEARCH = False
SEARCH_PROVIDER = "exa"  # or "tavily", "perplexity", "openai"
TOPIC = "What are the tradeoffs between passkeys and passwords for account recovery?"

if RUN_WEB_RESEARCH:
    web_agent = ResearchAgent(llm=BedrockLLM(), search=make_search(SEARCH_PROVIDER), cap=2)
    topic_run = Path("runs") / f"topic-{datetime.now():%Y%m%d-%H%M%S}"
    web_result = await web_agent.research(TOPIC, out=topic_run)
    print(f"Citation checks passed: {web_result.valid}; saved to {topic_run}")
    display(Markdown(web_result.markdown()))
else:
    print("Optional web research is disabled. Configure your providers, then set RUN_WEB_RESEARCH = True.")